# Four-Bar Linkage Animation — matplotlib starter
## ME 493B MP4 Part A — companion to the HTML starter

This notebook is for students who'd rather stay in Python than open a
browser tab. It does the same thing as `four_bar_rapier_starter.html`:
animate a planar four-bar linkage on one side of the MiniClaw gripper,
with an optional mirrored other side.

The kinematics here is **the same analytical solution** you'd use in
your notebook's `compute_finger_position()` function (Section 3) —
vector loop closure via two-circle intersection. Reading this code is
a perfectly legitimate way to start your Section 3 implementation.

### What to change for your design

1. Edit the `LINKS`, `OUTPUT_PIVOT`, `TIP_EXT`, and `INPUT_RANGE_DEG`
   constants in the next cell to match your geometry.
2. Toggle `SHOW_MIRROR` if you want to see the closing motion of both
   sides. The mirror axis is x = `MIRROR_AXIS_X` (default 0).
3. Run all cells. The animation should appear inline.

### How to capture results for the notebook

Save the animation as MP4 or GIF (the last cell shows how) and commit it
under `MP4/Part A/motion/`. Or take a sequence of screenshots at the
open / mid / closed positions.

In [ ]:
# ============================================================
# EDIT THESE TO MATCH YOUR DESIGN
# ============================================================
# Parallelogram four-bar inspired by the BigClaw (L1=L3, L2=L4).
# INPUT_PIVOT = (8, 0): your side's gear pivot, OFFSET from the gripper
# centerline (mirror axis at x=0). The X offset = half the gear pair
# center distance — each gear's center sits this far from where the
# gears mesh. With both sides mirrored, the two gear pivots end up at
# (+8, 0) and (-8, 0), a 16 mm center distance you'll design in Part B.
# O4=(8,12) is directly above O2, giving a vertical ground link of length
# 12 mm (L1=12) at phi=90°.
# Dead-center at theta=90°; the range 0°–50° stays safely below it.
# Transmission angle mu = |theta - 90°|, giving 90°–40° across this range
# (sits right at the 40° floor at theta_max — give yourself more margin
# when designing your own).
# The cranks (L2=L4=25 mm) are LONGER than the ground/coupler (L1=L3=12 mm)
# — that's how the BigClaw layout looks: short ground link, long finger.
# At theta_min=0° the finger is at max-open; at theta_max=50° it is closed.
LINKS = {
    "L1": 12.0,   # ground link (mm) = L3 (parallelogram)
    "L2": 25.0,   # input crank (mm) = L4 (parallelogram) — rotates with gear pivot
    "L3": 12.0,   # coupler     (mm) = L1 (parallelogram) — stays parallel to ground
    "L4": 25.0,   # output crank(mm) = L2 (parallelogram)
}
INPUT_PIVOT  = (8.0, 0.0)      # (x, y) of O2 in mm; offset from mirror axis = gear-pair-offset
OUTPUT_PIVOT = (8.0, 12.0)     # (x, y) of O4 in mm; directly above O2 (vertical ground link)
TIP_EXT = 35.0                 # finger-tip extension past joint B along the COUPLER, mm
INPUT_RANGE_DEG = (0.0, 50.0)  # (theta_min, theta_max) — open → closed for this geometry

SHOW_MIRROR    = True
MIRROR_AXIS_X  = 0.0           # gripper centerline; sits between the two mirrored sides

# ============================================================
# NO MORE CONSTANTS PAST THIS POINT (unless you want to)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

## The kinematics

Given input angle θ₂ (in radians), link lengths, the input ground pivot
O₂, and the output ground pivot O₄, find joint A and then joint B:

- A = O₂ + (L₂ cos θ₂, L₂ sin θ₂)
- B is the intersection of the circle of radius L₃ centered on A and
  the circle of radius L₄ centered on O₄

Two assembly modes exist; the `branch` argument picks one. **For the
default vertical (and most upper-left) ground link, use `branch=-1`** to
get the parallelogram-style "B sits above A" assembly that matches the
identity B = A + (O₄ − O₂). For an upper-right O₄, use `branch=+1`. If
your linkage renders inside-out, flip the sign.

The finger tip is computed as a rigid extension of the **coupler**
(NOT the output crank). For a parallelogram, the coupler stays parallel
to the ground link at every θ — so the finger translates without
rotating, giving parallel jaw motion. Attaching the tip to the output
crank (which the older starter did) made the finger swing in a big arc
— wrong for a parallel gripper.

**Note on frames.** This function takes O₂ as a parameter so the
visualization can place your side offset from the mirror axis (= the
gear pair offset; see the constants cell). Your notebook's
`compute_finger_position()` can stay simpler — use a *local* frame
where O₂ = (0, 0) and apply the offset only at integration time.
Either form yields identical kinematics.

In [ ]:
def four_bar_state(theta2_deg, links, input_pivot, output_pivot, branch=-1):
    """Return joint positions for a four-bar at a given input angle.

    Args:
        theta2_deg:   input link angle in degrees (measured CCW from +x)
        links:        dict with keys L1, L2, L3, L4 (L1 not used here but
                      kept in the dict so your `LINKS` is the single source
                      of truth)
        input_pivot:  (IX, IY) location of O2 in mm — your side's gear pivot,
                      offset along x from the gripper centerline (mirror axis)
        output_pivot: (OX, OY) location of O4 in mm
        branch:       -1 (default) for vertical / upper-left O4 — gives the
                      "B above A" parallelogram assembly. Use +1 for an
                      upper-right O4. If your visualization shows the
                      linkage assembling on the wrong side, flip the sign.

    Returns:
        dict with keys "ok", "O2", "A", "B", "O4", "mu_deg" (None if !ok)
    """
    L2, L3, L4 = links["L2"], links["L3"], links["L4"]
    IX, IY = input_pivot
    OX, OY = output_pivot
    t = np.radians(theta2_deg)
    Ax, Ay = IX + L2 * np.cos(t), IY + L2 * np.sin(t)
    dx, dy = OX - Ax, OY - Ay
    d = np.hypot(dx, dy)
    if d > L3 + L4 + 1e-9 or d < abs(L3 - L4) - 1e-9:
        return {"ok": False, "O2": (IX, IY), "A": (Ax, Ay), "O4": (OX, OY)}
    a = (d**2 + L3**2 - L4**2) / (2 * d)
    h = np.sqrt(max(0.0, L3**2 - a**2))
    fx, fy = Ax + a * dx / d, Ay + a * dy / d
    # perpendicular (rotate (dx,dy)/d by +90°)
    px, py = -dy / d, dx / d
    Bx, By = fx + branch * h * px, fy + branch * h * py
    # transmission angle = angle at B between coupler (B->A) and output (B->O4)
    v1 = np.array([Ax - Bx, Ay - By])
    v2 = np.array([OX - Bx, OY - By])
    cos_mu = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
    mu = np.degrees(np.arccos(np.clip(cos_mu, -1, 1)))
    return {"ok": True, "O2": (IX, IY), "A": (Ax, Ay), "B": (Bx, By),
            "O4": (OX, OY), "mu_deg": float(mu)}


def finger_tip(state, tip_ext):
    """Tip position = joint B extended by tip_ext along the COUPLER
    direction (from A toward B). For a parallelogram this direction is
    FIXED in the world frame (parallel to the ground link at every theta),
    so the finger translates without rotating — that is what makes the
    jaw motion parallel."""
    if not state["ok"]:
        return None
    Bx, By = state["B"]
    Ax, Ay = state["A"]
    ux, uy = Bx - Ax, By - Ay        # coupler direction (A -> B)
    m = np.hypot(ux, uy) or 1e-9
    return (Bx + tip_ext * ux / m, By + tip_ext * uy / m)


# quick smoke test at the open end of the input range
theta_test = INPUT_RANGE_DEG[0]
s0 = four_bar_state(theta_test, LINKS, INPUT_PIVOT, OUTPUT_PIVOT)
print(f"theta = {theta_test}: ok={s0['ok']}, A={s0['A']}, B={s0.get('B')}, mu={s0.get('mu_deg')}")
print(f"finger tip: {finger_tip(s0, TIP_EXT)}")

## Sweep the input range and animate

The animation steps θ₂ from `INPUT_RANGE_DEG[0]` to `[1]` and back, at
one degree per frame. Both sides are drawn if `SHOW_MIRROR = True`.

In [ ]:
def mirror_pt(p, axis_x):
    return (2 * axis_x - p[0], p[1])

fig, ax = plt.subplots(figsize=(7.5, 6.0))
ax.set_aspect("equal")
ax.set_facecolor("#1a1a1a")
fig.patch.set_facecolor("#111")
for spine in ax.spines.values():
    spine.set_color("#333")
ax.tick_params(colors="#888")

# Pre-compute viewport bounds across the sweep so the axes don't jump
pts = []
for th in np.linspace(*INPUT_RANGE_DEG, 60):
    s = four_bar_state(th, LINKS, INPUT_PIVOT, OUTPUT_PIVOT)
    if not s["ok"]: continue
    pts += [s["O2"], s["A"], s["B"], s["O4"]]
    tip = finger_tip(s, TIP_EXT)
    if tip is not None:
        pts.append(tip)
        if SHOW_MIRROR:
            pts.append(mirror_pt(tip, MIRROR_AXIS_X))
            pts += [mirror_pt(s["O2"], MIRROR_AXIS_X), mirror_pt(s["A"], MIRROR_AXIS_X),
                    mirror_pt(s["B"],  MIRROR_AXIS_X), mirror_pt(s["O4"], MIRROR_AXIS_X)]
if pts:
    xs, ys = zip(*pts)
    ax.set_xlim(min(xs) - 10, max(xs) + 10)
    ax.set_ylim(min(ys) - 10, max(ys) + 10)
ax.set_xlabel("x (mm)", color="#aaa")
ax.set_ylabel("y (mm)", color="#aaa")
title = ax.set_title("", color="#ddd", fontsize=11)

# Artists we'll update each frame
line_ground, = ax.plot([], [], color="#666", lw=2)
line_input,  = ax.plot([], [], color="#6cf", lw=3.5)
line_coupler,= ax.plot([], [], color="#fc6", lw=3.5)
line_output, = ax.plot([], [], color="#9f9", lw=3.5)
line_tip,    = ax.plot([], [], color="#bfb", lw=3.5)
pivots = ax.scatter([], [], c="#fff", s=40, zorder=5)
tip_dot, = ax.plot([], [], "o", color="#bfb", ms=7)
tip_trail, = ax.plot([], [], "-", color="#6cf", lw=0.8, alpha=0.5)

if SHOW_MIRROR:
    mline_input,   = ax.plot([], [], color="#fa6", lw=3, alpha=0.8)
    mline_coupler, = ax.plot([], [], color="#fc6", lw=3, alpha=0.6)
    mline_output,  = ax.plot([], [], color="#f99", lw=3, alpha=0.8)
    mline_tip,     = ax.plot([], [], color="#fbb", lw=3, alpha=0.8)
    mtip_dot,      = ax.plot([], [], "o", color="#fbb", ms=7)
    mtip_trail,    = ax.plot([], [], "-", color="#f99", lw=0.8, alpha=0.5)
    ax.axvline(MIRROR_AXIS_X, color="#446", linestyle=":", lw=1)

# state for the trails
trail_xy = []
mtrail_xy = []

def step_frame(theta_deg):
    s = four_bar_state(theta_deg, LINKS, INPUT_PIVOT, OUTPUT_PIVOT)
    if not s["ok"]:
        title.set_text(f"θ = {theta_deg:+.0f}°  —  linkage breaks (no assembly)")
        for a in (line_input, line_coupler, line_output, line_tip):
            a.set_data([], [])
        return
    O2, A, B, O4 = s["O2"], s["A"], s["B"], s["O4"]
    tip = finger_tip(s, TIP_EXT)
    line_ground.set_data([O2[0], O4[0]], [O2[1], O4[1]])
    line_input. set_data([O2[0], A[0]],  [O2[1], A[1]])
    line_coupler.set_data([A[0], B[0]],  [A[1], B[1]])
    line_output. set_data([O4[0], B[0]], [O4[1], B[1]])
    if tip is not None:
        line_tip.set_data([B[0], tip[0]], [B[1], tip[1]])
        tip_dot.set_data([tip[0]], [tip[1]])
        trail_xy.append(tip)
        if len(trail_xy) > 1500: del trail_xy[0]
        tx, ty = zip(*trail_xy); tip_trail.set_data(tx, ty)
    pivots.set_offsets([O2, A, B, O4])

    if SHOW_MIRROR:
        mO2 = mirror_pt(O2, MIRROR_AXIS_X)
        mA  = mirror_pt(A,  MIRROR_AXIS_X)
        mB  = mirror_pt(B,  MIRROR_AXIS_X)
        mO4 = mirror_pt(O4, MIRROR_AXIS_X)
        mline_input.  set_data([mO2[0], mA[0]], [mO2[1], mA[1]])
        mline_coupler.set_data([mA[0],  mB[0]], [mA[1],  mB[1]])
        mline_output. set_data([mO4[0], mB[0]], [mO4[1], mB[1]])
        if tip is not None:
            mtip = mirror_pt(tip, MIRROR_AXIS_X)
            mline_tip.set_data([mB[0], mtip[0]], [mB[1], mtip[1]])
            mtip_dot.set_data([mtip[0]], [mtip[1]])
            mtrail_xy.append(mtip)
            if len(mtrail_xy) > 1500: del mtrail_xy[0]
            mx, my = zip(*mtrail_xy); mtip_trail.set_data(mx, my)

    title.set_text(f"θ = {theta_deg:+.0f}°    μ = {s['mu_deg']:.1f}°")

# Frame schedule: forward then back (one full cycle)
frames_fwd = list(np.arange(INPUT_RANGE_DEG[0], INPUT_RANGE_DEG[1] + 1, 1.0))
frames_rev = list(reversed(frames_fwd))
frames = frames_fwd + frames_rev

anim = animation.FuncAnimation(fig, step_frame, frames=frames, interval=40, blit=False)
plt.close(fig)  # so we get the inline animation, not the static figure
HTML(anim.to_jshtml())

### Save the animation

Uncomment one of the lines below to save the animation as MP4 or GIF
and commit it under `MP4/Part A/motion/` for your Section 5 evidence.

- MP4 needs `ffmpeg` available on PATH. Codespaces usually has it.
- GIF needs `pillow`; `pip install pillow` if you don't have it.

In [ ]:
# anim.save("../motion/four_bar_sweep.mp4", fps=25)
# anim.save("../motion/four_bar_sweep.gif", writer="pillow", fps=20)
print("uncomment the line above to save the animation")

### What to take from here into your notebook

- `four_bar_state()` is essentially what your notebook's
  `compute_finger_position()` (Section 3) needs to return — adapt the
  output shape to `(tip_x, tip_y, displacement_from_closed)`. The
  notebook function signature drops `input_pivot` because Section 1
  uses a local frame where O₂ = (0, 0); you'd apply the gear-pair
  offset only when integrating into the full housing in Part B.
- `s["mu_deg"]` is what your notebook's `compute_transmission_angle()`
  (Section 4) needs to return.
- The animation here is the visible-motion artifact for Section 5; save
  it under `motion/` and reference it from the notebook.